In [15]:
import torch
import torch.nn as nn
import torch.onnx as onnx
import os
import json
import shutil
from random import randint
from ml_runner_exporter.onnx_exporter import export_onnx

In [16]:
fixtures_path = "tests/fixtures"

if os.path.exists(fixtures_path):
    shutil.rmtree(fixtures_path)
os.makedirs(fixtures_path)

In [17]:
def get_output_size() -> int:
    return randint(2, 10) * 5

In [18]:
class SimpleLinearModel(nn.Module):
    def __init__(self, input_size, output_size, layer_num):
        super(SimpleLinearModel, self).__init__()
        self.layers = nn.ModuleList()
        if layer_num == 1:
            self.layers.append(nn.Linear(input_size, output_size))
        else:
            inter_output_size = get_output_size()
            self.layers.append(nn.Linear(input_size, inter_output_size))
            inter_input_size = inter_output_size
            for i in range(layer_num - 2):
                inter_output_size = get_output_size()
                self.layers.append(nn.Linear(inter_input_size, inter_output_size))
                inter_input_size = inter_output_size
            self.layers.append(nn.Linear(inter_input_size, output_size))

    def forward(self, x):
        # Pass input through the linear layer
        output = x
        for layer in self.layers:
            output = layer.forward(output)
        return output

In [19]:
def export_model(name: str, model: nn.Module, input_size: int):
    tmp_model_path = "temporary_model.onnx"
    onnx.export(model, torch.randn(1, input_size), tmp_model_path, export_params=True, opset_version=11)
    dummy_input_data = torch.randn(1, input_size, dtype=torch.float32)
    with torch.no_grad():
        output = model(dummy_input_data)
    model_output = {
        "model": export_onnx(tmp_model_path),
        "test_input": dummy_input_data.tolist()[0],
        "test_output": output.tolist()[0],
    }

    with open(os.path.join(fixtures_path, name), "w") as f:
        print("Exporting model")
        json.dump(model_output, f, indent=2)

In [20]:
fixtures = [
    {
        "name": "dense_simple_model.json",
        "model": SimpleLinearModel(10, 5, 1),
        "input_size": 10,
    },
    {
        "name": "dense_long_model.json",
        "model": SimpleLinearModel(10, 5, 20),
        "input_size": 10,
    },
    {
        "name": "dense_large_model.json",
        "model": SimpleLinearModel(100, 100, 5),
        "input_size": 100,
    },
]

In [21]:
for fixture in fixtures:
    export_model(fixture["name"], fixture["model"], fixture["input_size"])

/tmp/ipykernel_54029/562235624.py:3: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  onnx.export(model, torch.randn(1, input_size), tmp_model_path, export_params=True, opset_version=11)
W0815 19:08:29.436000 54029 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
/home/david/miniconda3/envs/pytorch/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, us

[torch.onnx] Obtain model graph for `SimpleLinearModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleLinearModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exporting model
[torch.onnx] Obtain model graph for `SimpleLinearModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleLinearModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/david/miniconda3/envs/pytorch/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/david/miniconda3/envs/pytor

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exporting model
[torch.onnx] Obtain model graph for `SimpleLinearModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleLinearModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/david/miniconda3/envs/pytorch/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/home/david/miniconda3/envs/pytorch/lib/python3.14/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/david/miniconda3/envs/pytor

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exporting model
